# Facial Expression-Based Emotion Recognition

## Environment Preparation

### Package Importing

In [ ]:
# Basic imports
import os
import sys
import warnings
from dotenv import load_dotenv
# Data manipulation and numerical processing libraries
import numpy as np
import pandas as pd
from functools import partial

# Machine learning and evaluation tools
import psychai.facial_expression.feature_extraction.feature_extraction

import psychai.facial_expression

import psychai.facial_expression.feature_extraction

# Ignore warnings for cleaner output
warnings.filterwarnings("ignore")

# Custom feature extraction modules from psychai (assuming psychai is installed in your environment)
import psychai.feature.feature_extraction.feature_retriever
import psychai.feature.feature_extraction.feature_processor

# Initialize feature processor from psychai
feature_processor = psychai.feature.feature_extraction.feature_processor.FeatureProcessor()

from datetime import datetime

today = datetime.now().strftime('%Y%m%d')

# Set base directories and file paths

load_dotenv(override=True)
huggingface_cache_location = os.getenv("huggingface_cache_location")
datasets_cache_location = os.getenv("datasets_cache_location")
resutls_location = os.getenv("results_location")
resource_path= os.getenv("resources_location")
resources_location_external= os.getenv("resources_location_external")

raw_data_path =  os.path.join(resource_path, "examples", "paper5_mmer_moral","resources","raw_data")
csv_cache_file_path =  os.path.join(resutls_location, "examples", "paper5_mmer_moral", "facial",f"facial_features_cache_{today}.csv")
csv_cache_baseline_file_path =  os.path.join(resutls_location, "examples", "paper5_mmer_moral", "facial",f"facial_features_baseline_cache_{today}.csv")
pickle_file_path =  os.path.join(resutls_location, "examples", "paper5_mmer_moral", "facial",f"facial_features_cache_{today}.pkl")
pickle_baseline_file_path =  os.path.join(resutls_location, "examples", "paper5_mmer_moral", "facial",f"facial_features_baseline_cache_{today}.pkl")
pickle_folder_path =  os.path.join(resutls_location, "examples", "paper5_mmer_moral", "facial")

folder_path = os.path.join(resource_path, "examples", "paper5_mmer_moral","resources","raw_data","005","RecordedVideo")
#folder_path = '../resources/raw_data'

# Extend system path to current directory (for importing modules if needed)
current_dir = os.path.abspath('../')
sys.path.append(current_dir)

# Import any custom utilities from the parent directory if needed
import local_utilities

# Specify modality (e.g., audio segment type) for processing
modality_value = "Facial_Expression"

# Import the `Image` class from the `IPython.display` module to display images directly in Jupyter notebooks or IPython environments.
# This will allow images (e.g., facial expressions) to be shown inline for easier visualization and analysis.
from IPython.display import Image

# Import custom utility functions or classes from a local module called `local_utilities`. 
import local_utilities

features_extractor = psychai.facial_expression.feature_extraction.feature_extraction.FacialExpresionFeatureExtractor(
    landmark_model="mobilefacenet",
    au_model='xgb',
    emotion_model="resmasknet",
    identity_model="facenet",
    device="cuda",
)

## Single Image Example

In [ ]:
single_image_path =  os.path.join(resource_path, "examples", "paper5_mmer_moral", "resources","images","face_example.jpg")
Image(filename=single_image_path, width=200) 

In [ ]:
features, feature_names = features_extractor.process_image(single_image_path, show_frame_images=True, add_distance=True)
features

## Single Video Example

In [ ]:
files = os.listdir(folder_path)
files[0]
single_video_path = os.path.join(folder_path,files[0])
df_features = features_extractor.process_video(file_path=single_video_path, sampling_time=240, show_frame_images= False)
df_features

## Target Folder Processing

In [ ]:
single_analysis = local_utilities.SingleModalityAnalysis()
raw_data_path = '../resources/raw_data'
filters = [(2, '23')]
user_ids = range(1, 120)
participant_data_path =  os.path.join(resource_path, "examples", "paper5_mmer_moral", "resources","data","rct","rct.csv")
merging_pdf = pd.read_csv(participant_data_path)
df_largest = single_analysis.prosss_folder(raw_data_path, filters, user_ids, merging_pdf, modality_value="RecordedVideo")

My labtop takes about 30 seconds for one frame.
So if we take one frame per 5 seconds, than a 10 minutes video takes about 60 minutes to complete.
With parallel processing, it actually took 7 and half minitues to complete a 10 minutes video and
48 minutes to run 7 videos.


In [ ]:
pickle_file_path = '../results/facial_features.csv'
csv_cache_file_path = '../results/facial_features.csv'

In [ ]:
sampling_time_in_sec = 5
processor = psychai.feature.feature_extraction.feature_retriever.FeatureRetriever(df_largest, feature_extractors=[partial(features_extractor.process_video, sampling_time=sampling_time_in_sec, show_frame_images = False)]
                             ,cache_file_path=pickle_file_path)

# Extract features and return the final DataFrame
df_with_features = processor.extract_features(override= False)

In [ ]:
df_with_features.iloc[0:10,0:30]

In [ ]:
df_with_features.loc[:, df_with_features.columns.str.contains('AU')]

In [ ]:
df_with_features.to_csv(csv_cache_file_path)

In [ ]:
df_with_features["user_id"]

## Baseline Folder Processing

In [ ]:
single_analysis = local_utilities.SingleModalityAnalysis()
raw_data_path = '../resources/raw_data'
filters = [(2, '21')]
user_ids = range(1, 120)
participant_data_path =  os.path.join(resource_path, "examples", "paper5_mmer_moral", "resources","data","rct","rct.csv")
merging_pdf = pd.read_csv(participant_data_path)
df_largest_baseline = single_analysis.prosss_folder(raw_data_path, filters, user_ids, merging_pdf, modality_value="RecordedVideo")

In [ ]:
sampling_time_baseline_in_sec = 60
processor_baseline = psychai.feature.feature_extraction.feature_retriever.FeatureRetriever(df_largest_baseline, feature_extractors=[partial(features_extractor.process_video, sampling_time=sampling_time_baseline_in_sec, show_frame_images = False)]
                             ,cache_file_path=pickle_baseline_file_path)

# Extract features and return the final DataFrame
df_baseline_with_features = processor_baseline.extract_features(override= False)

In [ ]:
df_baseline_with_features

In [ ]:
df_baseline_with_features

#### To CSV

In [ ]:
df_baseline_with_features.to_csv(csv_cache_baseline_file_path)

## 

In [ ]:
df_summary = features_extractor.summarized_data(df_with_features)

## Footnote
- Copyright (c) 2026 Ivan Liu. Licensed under CC BY 4.0.
- Last Update: 2024
- env41204
- References: 